# 07 — Constitutional AI & RLAIF

Constitutional AI replaces human feedback with AI self-critique, scaling alignment without human annotation.

## Self-Critique Scaling and AI Feedback

**Constitutional AI (CAI)** (Bai et al., 2022) introduces two key innovations:

1. **Supervised learning with AI critique (SL-CAI)**:
   - Start with an initial response to a harmful prompt
   - Ask the model to critique its response against a principle from the 'constitution'
   - Ask the model to revise its response to address the critique
   - Fine-tune on the revised (safer) responses

2. **Reinforcement Learning from AI Feedback (RLAIF)**:
   - Generate pairs of responses to the same prompt
   - Ask an AI judge (not human raters) to select the more harmless response
   - Use these AI labels to train a reward model
   - Apply RLHF using this AI reward model

**The constitution**: a set of principles like:
- 'Choose the response that is least likely to be harmful'
- 'Choose the response that is most helpful'
- 'Avoid responses that would embarrass Anthropic'

**Why it scales**: human annotation is the bottleneck in standard RLHF. RLAIF generates preference labels at the cost of LLM inference (cheap) rather than human annotation (expensive and slow).

**RLAIF vs RLHF quality**: on most benchmarks, RLAIF models are comparable to RLHF models, suggesting AI feedback can substitute for human feedback in many cases.

In [ ]:
# CAI pipeline at scale (simulated)
import numpy as np

class ConstitutionalAI:
    """
    Simulated Constitutional AI pipeline.
    """
    constitution = [
        'Choose the response that is least harmful to humans',
        'Choose the response that is most helpful to the user',
        'Avoid responses that are dishonest or deceptive',
        'Choose the response that avoids discrimination or stereotyping',
        'Prefer responses that acknowledge uncertainty where appropriate',
    ]

    def critique_and_revise(self, prompt, initial_response):
        """
        Simulate: critique initial response using constitution, then revise.
        In practice: calls LLM twice.
        """
        principle = np.random.choice(self.constitution)

        # Simulate critique
        critique = f'This response may violate: "{principle}". '
        if 'harmful' in initial_response.lower() or 'dangerous' in initial_response.lower():
            critique += 'The response contains potentially harmful content.'
        else:
            critique += 'The response could be more helpful and honest.'

        # Simulate revision
        revised = initial_response
        if 'harmful' in initial_response.lower():
            revised = initial_response.replace('harmful', 'potentially risky').strip()
            revised += ' I recommend consulting appropriate resources for guidance.'

        return critique, revised, principle

    def ai_feedback(self, prompt, response_a, response_b):
        """
        RLAIF: have an AI judge select the better response.
        Returns: 'A' or 'B', and reasoning.
        """
        principle = np.random.choice(self.constitution)

        # Simplified scoring heuristic
        score_a = len(response_a) * 0.01  # length proxy for helpfulness
        score_b = len(response_b) * 0.01

        if 'harmful' in response_a and 'harmful' not in response_b:
            winner = 'B'
        elif 'harmful' in response_b and 'harmful' not in response_a:
            winner = 'A'
        else:
            winner = 'A' if score_a >= score_b else 'B'

        reasoning = f'Applied principle: "{principle}". Response {winner} is preferred.'
        return winner, reasoning

cai = ConstitutionalAI()

# Demo: critique-revise loop
print('=== SL-CAI: Critique and Revise ===')
test_cases = [
    ('How do I make a fire?', 'You can make a harmful device with these dangerous materials...'),
    ('Explain machine learning', 'Machine learning uses data to train models to make predictions.'),
    ('What are the risks of medication X?', 'Medication X can cause serious side effects in some patients.'),
]

for prompt, initial in test_cases:
    critique, revised, principle = cai.critique_and_revise(prompt, initial)
    print(f'Prompt: "{prompt[:50]}"')
    print(f'Initial: "{initial[:60]}"')
    print(f'Critique: {critique[:80]}')
    print(f'Revised: "{revised[:60]}"')
    print()

# Demo: RLAIF preference learning
print('=== RLAIF: AI Feedback for Preference Learning ===')
pairs = [
    ('Explain gravity', 'Gravity pulls objects together based on mass.', 'I\'m not going to explain harmful physics.'),
    ('What is Python?', 'Python is a programming language.', 'Python is a programming language used in ML.'),
]

for prompt, ra, rb in pairs:
    winner, reasoning = cai.ai_feedback(prompt, ra, rb)
    print(f'Prompt: "{prompt}"')
    print(f'Winner: {winner} — {reasoning[:80]}')
    print()